# Stage 1: Preprocessing & Phase Spectrum Extraction

**Project:** KAN-Driven Phase-Spectrum Analysis for Next-Generation Deepfake Detection  
**Environment:** Kaggle Notebook (GPU T4)  

This notebook implements:
1. Spatial standardisation (RGB → Grayscale → 256×256 bicubic resize)
2. 2D FFT with frequency centering
3. Phase extraction & min-max normalisation to [0, 1]
4. Side-by-side visualisation of all intermediate outputs

---
## Cell 1: Imports

In [ ]:
# =============================================================================
# Cell 1: Imports
# =============================================================================
import numpy as np
import cv2
from skimage import img_as_float
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from typing import Dict, Tuple, Optional

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9

# --- Kaggle Paths ---
INPUT_DIR = '/kaggle/input/artifact-dataset'
OUTPUT_DIR = '/kaggle/working'

print('All imports successful.')
print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'Dataset: {INPUT_DIR}')
print(f'Output : {OUTPUT_DIR}')

---
## Cell 2: Class & Function Definitions

In [ ]:
# =============================================================================
# Cell 2: Class & Function Definitions
# =============================================================================


class PhaseSpectrumExtractor:
    """
    Extracts the normalised phase spectrum from an input image.
    
    Pipeline:
        1. Load RGB image
        2. Convert to grayscale (luminance-preserving)
        3. Resize to target dimensions using bicubic interpolation
        4. Compute 2D FFT and centre the zero-frequency component
        5. Extract phase angle via arctan2
        6. Min-max normalise phase from [-pi, pi] to [0, 1]
    """
    _LUMA_R: float = 0.2989
    _LUMA_G: float = 0.5870
    _LUMA_B: float = 0.1140
    
    def __init__(self, target_size: Tuple[int, int] = (256, 256)) -> None:
        self.target_size = target_size
    
    def load_image(self, image_path: str) -> np.ndarray:
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f'Image not found: {image_path}')
        bgr_image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if bgr_image is None:
            raise ValueError(f'Could not decode: {image_path}')
        return cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    
    def rgb_to_grayscale(self, rgb_image: np.ndarray) -> np.ndarray:
        img_float = rgb_image.astype(np.float64)
        return (self._LUMA_R * img_float[:,:,0] +
                self._LUMA_G * img_float[:,:,1] +
                self._LUMA_B * img_float[:,:,2])
    
    def resize(self, image: np.ndarray) -> np.ndarray:
        return cv2.resize(image, (self.target_size[1], self.target_size[0]),
                          interpolation=cv2.INTER_CUBIC)
    
    def compute_fft(self, grayscale_image: np.ndarray) -> np.ndarray:
        return np.fft.fftshift(np.fft.fft2(grayscale_image))
    
    def extract_phase(self, fft_shifted: np.ndarray) -> np.ndarray:
        return np.angle(fft_shifted)
    
    def normalize_phase(self, phase: np.ndarray) -> np.ndarray:
        p_min, p_max = phase.min(), phase.max()
        if p_max - p_min == 0:
            return np.zeros_like(phase)
        return (phase - p_min) / (p_max - p_min)
    
    def extract_magnitude(self, fft_shifted: np.ndarray) -> np.ndarray:
        return np.log1p(np.abs(fft_shifted))
    
    def process(self, image_path: str) -> Dict[str, np.ndarray]:
        print(f'Processing: {os.path.basename(image_path)}')
        rgb = self.load_image(image_path)
        print(f'  [1/6] Loaded: {rgb.shape}')
        gray = self.rgb_to_grayscale(rgb)
        print(f'  [2/6] Grayscale: range=[{gray.min():.1f}, {gray.max():.1f}]')
        gray_resized = self.resize(gray)
        print(f'  [3/6] Resized: {gray_resized.shape}')
        fft_shifted = self.compute_fft(gray_resized)
        print(f'  [4/6] FFT: {fft_shifted.dtype}')
        phase_raw = self.extract_phase(fft_shifted)
        print(f'  [5/6] Phase: [{phase_raw.min():.4f}, {phase_raw.max():.4f}]')
        phase_norm = self.normalize_phase(phase_raw)
        print(f'  [6/6] Normalised: [{phase_norm.min():.4f}, {phase_norm.max():.4f}]')
        magnitude_log = self.extract_magnitude(fft_shifted)
        print('Pipeline complete.')
        return {
            'original_rgb': rgb, 'grayscale': gray,
            'grayscale_resized': gray_resized, 'fft_shifted': fft_shifted,
            'magnitude_log': magnitude_log, 'phase_raw': phase_raw,
            'phase_normalized': phase_norm,
        }


def visualize_results(results, save_path=None, figsize=(18, 4)):
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    fig.suptitle('Stage 1: Phase Spectrum Extraction Pipeline',
                 fontsize=14, fontweight='bold', y=1.02)
    axes[0].imshow(results['original_rgb'])
    axes[0].set_title('(a) Original RGB'); axes[0].axis('off')
    axes[1].imshow(results['grayscale_resized'], cmap='gray')
    axes[1].set_title('(b) Grayscale 256x256'); axes[1].axis('off')
    mag_plot = axes[2].imshow(results['magnitude_log'], cmap='inferno')
    axes[2].set_title('(c) FFT Magnitude (log)'); axes[2].axis('off')
    fig.colorbar(mag_plot, ax=axes[2], fraction=0.046, pad=0.04)
    phase_plot = axes[3].imshow(results['phase_normalized'], cmap='twilight', vmin=0, vmax=1)
    axes[3].set_title('(d) Phase Map [0, 1]'); axes[3].axis('off')
    fig.colorbar(phase_plot, ax=axes[3], fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print('PhaseSpectrumExtractor and visualize_results() defined.')

---
## Cell 3: Execution and Visualization

In [ ]:
# =============================================================================
# Cell 3: Execution and Visualization
# =============================================================================

# Find a sample image from the dataset
sample_image = None
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            sample_image = os.path.join(root, f)
            break
    if sample_image:
        break

if sample_image is None:
    raise FileNotFoundError('No images found in dataset!')

print(f'Using sample: {sample_image}')

# Run pipeline
extractor = PhaseSpectrumExtractor(target_size=(256, 256))
results = extractor.process(sample_image)

# Sanity checks
print(f'\n-- Sanity Checks --')
print(f'  Shape: {results["phase_normalized"].shape}')
print(f'  Range: [{results["phase_normalized"].min():.6f}, {results["phase_normalized"].max():.6f}]')
assert results['phase_normalized'].shape == (256, 256)
assert 0.0 <= results['phase_normalized'].min()
assert results['phase_normalized'].max() <= 1.0
print('  All assertions passed.')

visualize_results(results)